In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import yfinance as yf

stock_name = "LAURUSLABS.NS"

df = yf.download(stock_name, start="2020-01-01", end="2026-04-17")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)
df = df.dropna(subset=['Open', 'High', 'Low', 'Close', 'Volume'])
if len(df) < 120:
    raise ValueError('At least 120 valid price rows are required')
stocks = [stock_name]
data = {stock_name: df}
df.tail()

In [ ]:
close_prices = pd.DataFrame()

for stock in stocks:
    close_prices[stock] = data[stock]["Close"]

In [ ]:
close_prices.ffill(inplace=True)
close_prices.dropna(inplace=True)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

stock_index = stocks.index(stock_name)

stock_name = stocks[stock_index]

stock_data = data[stock_name][["Open", "High", "Low", "Close", "Volume"]].values


In [ ]:
split = int(0.8 * len(stock_data))

train_data = stock_data[:split]
test_data = stock_data[split:]

In [ ]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)

In [ ]:
test_scaled = scaler.transform(test_data)
scaled_data = np.vstack((train_scaled, test_scaled))

In [ ]:
X = []
y = []

for i in range(60, len(scaled_data)):
    X.append(scaled_data[i-60:i])
    y.append([scaled_data[i][0], scaled_data[i][3]])

X = np.array(X)
y = np.array(y)

In [ ]:
n_train = split - 60
X_train, X_test = X[:n_train], X[n_train:]
y_train, y_test = y[:n_train], y[n_train:]

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

In [ ]:
lstm_model = Sequential()

lstm_model.add(LSTM(128, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
lstm_model.add(Dropout(0.05))

lstm_model.add(LSTM(256, return_sequences=True))  # ✅ keep True
lstm_model.add(Dropout(0.05))

lstm_model.add(LSTM(128)) 
lstm_model.add(Dropout(0.05))

lstm_model.add(Dense(2))

lstm_model.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.15, shuffle=False,
    callbacks=[early_stop]
)

In [ ]:
lstm_predictions = lstm_model.predict(X_test)

In [ ]:
gru_model = Sequential()

gru_model.add(GRU(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
gru_model.add(Dropout(0.2))

gru_model.add(GRU(64))
gru_model.add(Dropout(0.2))

gru_model.add(Dense(2))

gru_model.compile(optimizer='adam', loss='mean_squared_error')

gru_model.summary()

In [ ]:
gru_history = gru_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.15, shuffle=False,
    # callbacks=[early_stop]
)

In [ ]:
gru_predictions = gru_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error

lstm_rmse = np.sqrt(mean_squared_error(y_test, lstm_predictions))
gru_rmse = np.sqrt(mean_squared_error(y_test, gru_predictions))

print("LSTM RMSE:", lstm_rmse)
print("GRU RMSE:", gru_rmse)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(y_test, label="Actual")
plt.plot(lstm_predictions, label="LSTM")
plt.plot(gru_predictions, label="GRU")

plt.title(f"LSTM vs GRU Prediction ({stock_name})")
plt.legend()
plt.show()

In [ ]:
# LSTM predictions
# LSTM
dummy_lstm = np.zeros((len(lstm_predictions), 5))
dummy_lstm[:, 0] = lstm_predictions[:, 0]
dummy_lstm[:, 3] = lstm_predictions[:, 1]

pred_lstm_real = scaler.inverse_transform(dummy_lstm)

open_pred_lstm = pred_lstm_real[:, 0]
close_pred_lstm = pred_lstm_real[:, 3]

# GRU predictions
dummy_gru = np.zeros((len(gru_predictions), 5))
dummy_gru[:, 0] = gru_predictions[:, 0]
dummy_gru[:, 3] = gru_predictions[:, 1]

pred_gru_real = scaler.inverse_transform(dummy_gru)

open_pred_gru = pred_gru_real[:, 0]
close_pred_gru = pred_gru_real[:, 3]


# Actual values
# Create dummy
dummy_y = np.zeros((len(y_test), 5))

# Assign correctly
dummy_y[:, 0] = y_test[:, 0]  # Open
dummy_y[:, 3] = y_test[:, 1]  # Close

# Inverse transform
y_real = scaler.inverse_transform(dummy_y)

# Extract
open_actual = y_real[:, 0]
close_actual = y_real[:, 3]

y_test_real = scaler.inverse_transform(dummy_y)[:, 3]

In [ ]:
plt.plot(close_actual, label="Actual Close")
plt.plot(close_pred_lstm, label="LSTM Close")
plt.plot(close_pred_gru, label="GRU Close")

In [ ]:
plt.plot(open_actual, label="Actual Open")
plt.plot(open_pred_lstm, label="LSTM Open")
plt.plot(open_pred_gru, label="GRU Open")

In [ ]:
print("Train loss:", lstm_history.history['loss'][-1])
print("Val loss:", lstm_history.history['val_loss'][-1])

In [ ]:
# Take last 60 days (already scaled OHLCV)
last_60_days = scaled_data[-60:]

# Reshape for model
last_60_days = last_60_days.reshape(1, 60, 5)

In [ ]:
# LSTM prediction
next_pred_lstm = lstm_model.predict(last_60_days)

# GRU prediction
next_pred_gru = gru_model.predict(last_60_days)

In [ ]:
dummy_lstm = np.zeros((1, 5))

dummy_lstm[0, 0] = next_pred_lstm[0, 0]  # Open
dummy_lstm[0, 3] = next_pred_lstm[0, 1]  # Close

next_lstm_real = scaler.inverse_transform(dummy_lstm)

lstm_open = next_lstm_real[0, 0]
lstm_close = next_lstm_real[0, 3]

In [ ]:
dummy_gru = np.zeros((1, 5))

dummy_gru[0, 0] = next_pred_gru[0, 0]
dummy_gru[0, 3] = next_pred_gru[0, 1]

next_gru_real = scaler.inverse_transform(dummy_gru)

gru_open = next_gru_real[0, 0]
gru_close = next_gru_real[0, 3]

In [ ]:
print("📊 NEXT DAY PREDICTION")

print(f"LSTM → Open: ₹{lstm_open:.2f}, Close: ₹{lstm_close:.2f}")
print(f"GRU  → Open: ₹{gru_open:.2f}, Close: ₹{gru_close:.2f}")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluate_model(y_true, y_pred, name="Model"):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)

    print(f"\n{name} Performance:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE : {mae:.4f}")
    print(f"R²  : {r2:.4f}")

    return rmse, mae, r2

# ✅ USE CORRECT VARIABLES
lstm_metrics = evaluate_model(y_test, lstm_predictions, "LSTM")
gru_metrics  = evaluate_model(y_test, gru_predictions, "GRU")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

plt.plot(y_test[:,1], label="Actual Close")
plt.plot(lstm_predictions[:,1], label="LSTM Close")
plt.plot(gru_predictions[:,1], label="GRU Close")

plt.title("Actual vs Predicted (Close Price)")
plt.legend()
plt.show()

In [ ]:
lstm_residuals = y_test[:,1] - lstm_predictions[:,1]
gru_residuals  = y_test[:,1] - gru_predictions[:,1]

plt.figure(figsize=(10,5))

plt.hist(lstm_residuals, bins=50, alpha=0.6, label="LSTM Errors")
plt.hist(gru_residuals, bins=50, alpha=0.6, label="GRU Errors")

plt.title("Error Distribution")
plt.legend()
plt.show()

In [ ]:
def direction_accuracy(y_true, y_pred):
    actual_dir = np.sign(y_true[:,1] - y_true[:,0])
    pred_dir   = np.sign(y_pred[:,1] - y_pred[:,0])

    return np.mean(actual_dir == pred_dir)

print("LSTM Direction Accuracy:", direction_accuracy(y_real[:, [0, 3]], pred_lstm_real[:, [0, 3]]))
print("GRU Direction Accuracy :", direction_accuracy(y_real[:, [0, 3]], pred_gru_real[:, [0, 3]]))